# All-atom polystyrene melt: MuPT → flowerMD AA-DPD → MuPT → OpenMM

This notebook builds a polystyrene (PS) melt and runs a short OpenMM simulation of it:

| Step | Tool | What happens |
|---|---|---|
| 1–3 | MuPT | build styrene repeat units from SMILES and link them into chains |
| 4 | MuPT → mBuild | export the atoms (with their MuPT addresses) |
| 5 | flowerMD | pack the chains at melt density and relax them with AA-DPD, keeping every stereocenter |
| 6 | mBuild → MuPT | write the relaxed coordinates back onto the MuPT representation |
| 7 | MuPT → MDAnalysis | write a PDB with chains and residues |
| 8 | OpenFF Interchange | assign the OpenFF Sage force field |
| 9 | OpenMM | minimize and run a few picoseconds of MD |

It follows the same steps as [`01_pe_all_atom_dpd`](./01_pe_all_atom_dpd.ipynb); start there for more explanation of each step. Polystyrene adds two things: a bulky side group (a phenyl ring per repeat unit) and a stereocenter in every repeat unit.

**Environment:** `pixi run -e workflows lab` or `conda-envs/workflows.yml`. Takes a few minutes on a laptop CPU.

> This uses the refactored MuPT API that is still under review (MuPT-hub/mupt#141 through #166), so names may still change before release.

In [ ]:
from networkx import connected_components

from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.connection.alignment import ConnectorAntialignmentRigid
from mupt.mupr.primitives import CompositePrimitive, RootPrimitive
from mupt.mupr.primitives.predicates import is_atom

## 1. System size

In [ ]:
num_chains: int = 10
DOP: int = 20  # repeat units per chain

## 2. Styrene repeat units from SMILES

`[*:1]` and `[*:2]` mark the bonds to neighboring repeat units; the chain ends are capped with hydrogen.
The backbone CH carrying the phenyl ring is a stereocenter. The SMILES leave its configuration open, so it is set by the embedded coordinates of the template. Every middle repeat unit is a copy of one template, placed by rigid moves that keep handedness, so they all have the same configuration: the chains are **isotactic** (the head unit comes from its own template). flowerMD keeps every configuration through the relaxation.

In [ ]:
lexicon = {
    "head": primitive_from_smiles("[H]-[CH2]-[CH:1](c1ccccc1)-[*:2]", label="head", embed_positions=True),
    "mid":  primitive_from_smiles("[*:1]-[CH2:2][CH:1](c1ccccc1)-[*:2]", label="mid", embed_positions=True),
    "tail": primitive_from_smiles("[*:1]-[CH2:2][CH2]c1ccccc1", label="tail", embed_positions=True),
}
sequence = ["head"] + (DOP - 2) * ["mid"] + ["tail"]

## 3. Link repeat units into chains

Same place-then-link loop as the PE example: place each repeat unit on its neighbor's free Connector, bond them, and tag chains as `segment` and repeat units as `residue`.

In [ ]:
def build_chain(sequence, bond_length=1.54):
    # Place each embedded repeat unit on its neighbor's Connector, then bond them.
    chain = CompositePrimitive(label="PS", role="segment")
    units = [lexicon[name].copy() for name in sequence]
    for unit in units:
        unit.role = "residue"
        chain.attach_child(unit)

    align = ConnectorAntialignmentRigid()
    for a, b in zip(units, units[1:]):
        x, y = next(
            (x, y)
            for x in a.connections.connectors_free
            for y in b.connections.connectors_free
            if x.bondable_with(y)
        )
        x.set_bond_length(bond_length)
        y.set_bond_length(bond_length)
        b.rigidly_transform(align.antialignment_transformation(align_connector=y, to_connector=x))
        a.connect_neighbor(b, our_connector=x, their_connector=y)
    return chain


system = RootPrimitive(label="PS melt", role="universe")
for _ in range(num_chains):
    system.attach_child(build_chain(sequence))

xsec = system.cross_section(is_atom)
print(f"{len(list(connected_components(xsec)))} chains, {xsec.number_of_nodes()} atoms, {xsec.number_of_edges()} bonds")

## 4. Export to mBuild

In [ ]:
from mupt.interfaces.mosdef.mbuild import primitive_to_mbuild

compound = primitive_to_mbuild(
    system,
    predicate=is_atom,
    preserve_intermediates=True,  # keep chain -> residue -> atom nesting in the Compound
    coords_to_nm=0.1,  # MuPT coordinates are in Angstrom; mBuild/flowerMD use nm
)
print(compound)

## 5. Pack and relax the melt with flowerMD AA-DPD

`OpenFF_DPD` builds the soft DPD forces from the OpenFF Sage force field. `PhantomWalk` then runs DPD until no atoms overlap, checks that no chain passes through a phenyl ring, and checks that every stereocenter still has its starting configuration.

In [ ]:
import warnings

import numpy as np
import unyt as u
from flowermd.base import Molecule
from flowermd.library import AtomisticRandomWalk, OpenFF_DPD, PhantomWalk

warnings.filterwarnings("ignore")

walk = AtomisticRandomWalk(
    [Molecule(num_mols=1, compound=chain) for chain in compound.children],  # one Molecule per MuPT chain
    density=0.97 * u.g / u.cm**3,
    seed=1,
)
walk.apply_forcefield(r_cut=0.35, force_field=OpenFF_DPD(walk), kT=4.184)
print("atoms:", walk.n_particles, " box (nm):", walk.box.Lx)

In [ ]:
sim = PhantomWalk(
    walk.hoomd_snapshot,
    walk.hoomd_forcefield,
    n_steps_dpd=40000,
    n_steps_fire=100,
    dt=2e-4,
    gsd_write_freq=int(1e5),
    log_write_freq=int(1e5),
    run_now=False,
)
record = sim.run_initialization(record_file="ps_record.json")
print("DPD converged:", record["dpd_converged"], "after", record["dpd_steps"], "steps")
print("ring piercings:", record["ring_piercings"])
print("stereochemistry:", record["stereochemistry"])

## 6. Put the relaxed coordinates back on the MuPT representation

Unwrap the periodic positions with each atom's box `image`, put them on the exported Compound, and let `update_primitive_from_mbuild` write them onto the matching Primitives by address.

In [ ]:
from mupt.interfaces.mosdef.mbuild import update_primitive_from_mbuild

snapshot = sim.state.get_snapshot()
box_nm = np.asarray(snapshot.configuration.box[:3])
compound.xyz = snapshot.particles.position + snapshot.particles.image * box_nm
moved = update_primitive_from_mbuild(system, compound, coords_to_nm=0.1)
print(f"updated {len(moved)} atoms on the MuPT representation")

## 7. Write a PDB from MuPT

In [ ]:
from mupt.interfaces.mdanalysis import primitive_to_mdanalysis

universe = primitive_to_mdanalysis(system, {"head": "STH", "mid": "STY", "tail": "STT"})
universe.dimensions = [*(box_nm * 10), 90.0, 90.0, 90.0]  # Angstrom
universe.atoms.write("ps_melt.pdb")

bond_lengths = np.linalg.norm(
    universe.atoms.positions[universe.bonds.indices[:, 0]] - universe.atoms.positions[universe.bonds.indices[:, 1]],
    axis=1,
)
print(f"{universe.atoms.n_atoms} atoms, {universe.residues.n_residues} residues, {universe.segments.n_segments} chains")
print(f"bond lengths {bond_lengths.min():.2f}-{bond_lengths.max():.2f} A")

## 8. Assign the OpenFF Sage force field

Each MuPT chain reaches OpenFF as an RDKit molecule (via mBuild, until MuPT's own RDKit exporter is ported to the refactored API). Positions and box come from the MuPT representation.

> Gasteiger charges keep this example fast. Use AM1-BCC (OpenFF's default) or NAGL charges for production.

In [ ]:
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule as OFFMolecule, Topology
from openff.units import unit
from rdkit import Chem

off_molecules = []
for chain in system.children:
    rdmol = primitive_to_mbuild(chain, predicate=is_atom, coords_to_nm=0.1).to_rdkit()
    Chem.SanitizeMol(rdmol)
    molecule = OFFMolecule.from_rdkit(rdmol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
    molecule.assign_partial_charges("gasteiger")
    off_molecules.append(molecule)

unique_molecules = []  # charge_from_molecules wants each distinct molecule once
for molecule in off_molecules:
    if not any(molecule.is_isomorphic_with(seen) for seen in unique_molecules):
        unique_molecules.append(molecule)

topology = Topology.from_molecules(off_molecules)
topology.box_vectors = box_nm * unit.nanometer
interchange = Interchange.from_smirnoff(
    ForceField("openff-2.3.0.offxml"),
    topology,
    charge_from_molecules=unique_molecules,
)
interchange.positions = universe.atoms.positions * unit.angstrom  # same atom order as the MuPT export
interchange.to_gromacs("ps_melt")  # also writes GROMACS inputs (ps_melt.gro / ps_melt.top)
print(interchange.topology.n_atoms, "atoms parameterized with", len(interchange.collections), "handlers")

## 9. Minimize and run a short OpenMM simulation

A Langevin integrator at 450 K (above polystyrene's glass transition) with a 1 fs step. This is a smoke test that the system is well formed, not an equilibration.

In [ ]:
import openmm
from openmm import unit as omm_unit

integrator = openmm.LangevinMiddleIntegrator(450 * omm_unit.kelvin, 1 / omm_unit.picosecond, 1 * omm_unit.femtosecond)
simulation = interchange.to_openmm_simulation(integrator)

def potential_energy():
    state = simulation.context.getState(getEnergy=True)
    return state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)

print(f"potential energy, start:     {potential_energy():12.1f} kJ/mol")
simulation.minimizeEnergy(maxIterations=500)
print(f"potential energy, minimized: {potential_energy():12.1f} kJ/mol")

simulation.context.setVelocitiesToTemperature(450 * omm_unit.kelvin, 1)
simulation.step(2000)  # 2 ps
state = simulation.context.getState(getEnergy=True, getPositions=True)
n_dof = 3 * simulation.system.getNumParticles() - simulation.system.getNumConstraints()
temperature = (2 * state.getKineticEnergy() / (n_dof * omm_unit.MOLAR_GAS_CONSTANT_R)).value_in_unit(omm_unit.kelvin)
print(f"potential energy, after 2 ps: {potential_energy():12.1f} kJ/mol")
print(f"temperature after 2 ps: {temperature:.0f} K")